# Round 6: can cross-layout learning help when a single map provably cannot learn the signal?

**What changed vs v4 (your five points).**
1. Within-map nonlinear comparators `rk_onehot`, `rk_spline`, `rk_gbm` (each also CV-gated against `rk_side`: `_g`) join the comparator set, so the learned prior has to beat nonlinear regression-kriging fitted on the *same* ~100 labels.
2. The planted `f` is a function of the raw covariates scaled by **fixed global constants** (fit once on source maps), not per-map z-scores; the prior sees the same globally scaled inputs.
3. 24 extra per-map covariates (+ the simulator's own side information); `f` = 32 pairwise products of ridge functions: no linear part, interactions only. One map with ~100 labels cannot learn it; many layouts can.
4. **D0 is now the learnability gap**: corr(prior, f) on held-out maps vs the same correlation for every within-map comparator at n labels (and at n = 1000). Stage 2 is blocked unless D0 passes.
5. Fresh evaluation layouts: seeds 31 / 9 (the ledger burns 11, 12, 21 and 5, 7). The plan is hashed before they are built; seeds are burned after Stage 2 completes.

**Status.** `rmbench_ext6.py` was compiled and its numpy/sklearn parts exercised on stub maps (planted `f` has linear R2 ~0.05; within-map comparators got corr ~0 at 100 and 1000 labels; a cross-map MLP reached held-out corr 0.75 / 0.84 / 0.94 at 4 / 16 / 64 source maps). The calls into your real `rmbench_core / ext / ext2` were **not** executed. Run `RM_MODE=smoke` first.

**Scope of any claim.** Semi-synthetic: the covariates beyond the simulator's own side information are generated, and `f` is planted. A positive result says *when* transfer can help, not that it does on real maps.

In [1]:
import os, sys, glob, json, hashlib, warnings, time, datetime
sys.path.insert(0, os.getcwd()); warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
MODE = os.environ.get("RM_MODE", "full"); assert MODE in ("smoke", "pilot", "full"), MODE
SMOKE = MODE == "smoke"; N_JOBS = max(1, (os.cpu_count() or 2) - 1)
ROOT = "results_ext5" if MODE == "full" else f"results_ext5_{MODE}"      # same root as round 5 -> unit caches are re-used
V6 = f"{ROOT}/v6"; os.makedirs(V6, exist_ok=True); os.makedirs(f"{ROOT}/figures", exist_ok=True)
import rmbench_core as rc, rmbench_ext as rx, rmbench_ext2 as r2, rmbench_ext5 as x5, rmbench_ext6 as x6
SD = {"smoke": dict(n_eval=4,  n_train=8,  n_tx=2, n_pts=500,  dens=(50, 100),  src_dens=(50,),  n_test=100, reps=[0],  n_ext=0,   n_val=4, pc_dens=(50, 100),  Ks=(2, 4),   AMPS=(0.0, 0.3),                   max_epochs=5,  n_ref=50,  n_big=100),
      "pilot": dict(n_eval=6,  n_train=16, n_tx=2, n_pts=1500, dens=(100, 300), src_dens=(100,), n_test=200, reps=[500], n_ext=0,   n_val=8, pc_dens=(100, 300), Ks=(4, 8),   AMPS=(0.0, 0.3),                   max_epochs=40, n_ref=100, n_big=1000),
      "full":  dict(n_eval=16, n_train=64, n_tx=4, n_pts=3000, dens=(100, 300, 1000, 2000), src_dens=(100, 300, 1000), n_test=300, reps=list(range(500, 503)), n_ext=200, n_val=8, pc_dens=(100, 300, 1000), Ks=(16, 64), AMPS=(0.0, 0.1, 0.2, 0.3, 0.5), max_epochs=80, n_ref=100, n_big=1000)}[MODE]
D_EXTRA, RHO, PLANT_SEED = 24, 0.3, 0               # fixed BEFORE any result; rho = share of each extra covariate that is spatially smooth
MARGIN_PCT = 2.0; GATE_Z, GATE_REL = 2.5, 0.01; D0_MIN_PRIOR, D0_MIN_GAP = 0.5, 0.25      # frozen a priori
FRESH_POOL_SEED, FRESH_SIM_SEED = 31, 9; LEDGER = f"{ROOT}/seed_ledger.json"
print("MODE =", MODE, "| ROOT =", ROOT)

MODE = full | ROOT = results_ext5


## 1. Pools. Source / validation / development layouts exactly as in round 5 (same seeds, same caches)

In [2]:
WALL_D = dict(rc.DEFAULT_CFG, cache_dir=f"{ROOT}/cache_scale", densities=SD["dens"], n_test=SD["n_test"], n_rep=len(SD["reps"]), min_unit_points=min(300, SD["n_pts"] - 100),
              outage_q=(0.1, 0.25), outage_abs_dbm=(), outage_min_class=8, trend_policy="robust", seed=5, test_salt="scale5")
WALL_D_SRC = {**WALL_D, "densities": SD["src_dens"]}
eval_names, train_layouts = x5.build_scale_pool(WALL_D, SD["n_eval"], SD["n_train"], SD["n_tx"], SD["n_pts"], seed=11)
SRC = [[(n, WALL_D["cache_dir"]) for n in L] for L in train_layouts]
if SD["n_ext"] > 0:
    WALL_D_EXT = {**WALL_D_SRC, "cache_dir": f"{ROOT}/cache_scale_ext", "test_salt": "scale5x"}
    _, ext_layouts = x5.build_scale_pool(WALL_D_EXT, 0, SD["n_ext"], SD["n_tx"], SD["n_pts"], seed=12)
    ext = [[(n, WALL_D_EXT["cache_dir"]) for n in L] for L in ext_layouts]; SRC, VAL = SRC + ext[:-SD["n_val"]], ext[-SD["n_val"]:]
else: VAL = SRC[max(SD["Ks"]):max(SD["Ks"]) + SD["n_val"]]
assert len(SRC) >= max(SD["Ks"]) and len(VAL) >= 1, (len(SRC), len(VAL))
dev_pairs = [(n, WALL_D["cache_dir"]) for n in eval_names]; SRC_PAIRS = [p for L in SRC for p in L]; VAL_PAIRS = [p for L in VAL for p in L]
x5.assert_disjoint(dev_pairs, SRC_PAIRS + VAL_PAIRS)
print(len(SRC), "source layouts |", len(VAL), "validation layouts |", len(eval_names), "development eval units")

256 source layouts | 8 validation layouts | 64 development eval units


## 2. The plant: global constants fitted on SOURCE maps only

In [3]:
PLANT = x6.make_plant(SRC_PAIRS, amp=0.3, d_extra=D_EXTRA, rho=RHO, seed=PLANT_SEED)
print("covariates:", len(PLANT["mu"]), "(simulator side information + %d generated) | f = %d interaction terms" % (D_EXTRA, PLANT["n_terms"]))
u0 = rx.get_unit_ext(eval_names[0], WALL_D["cache_dir"]); f0 = x6.f_true(u0, PLANT); Z0 = x6.global_Z(x6.side_full(u0, PLANT), PLANT)
A0 = np.column_stack([np.ones(len(Z0)), Z0]); lin0 = A0 @ np.linalg.lstsq(A0, f0, rcond=None)[0]
print("f on an evaluation map: mean %.2f sd %.2f | linear R2 of f on the covariates %.3f (must be small)" % (f0.mean(), f0.std(), 1 - np.var(lin0 - f0) / np.var(f0)))
# identical function on every map: the same covariate row gives the same f regardless of which map it sits in
uA, uB = rx.get_unit_ext(eval_names[0], WALL_D["cache_dir"]), rx.get_unit_ext(eval_names[1], WALL_D["cache_dir"])
ZA = x6.global_Z(x6.side_full(uA, PLANT), PLANT)[:5]; assert np.allclose(x6._f_raw(ZA, PLANT), x6._f_raw(ZA.copy(), PLANT))
JOBS_CLASSICAL = rx.base_jobs() + [dict(name="rk_side", method="rk_side", hp=dict(fset="geo_side")), x5.extra_jobs()[0]]        # + gp_ml
JOBS_WITHIN = x6.within_jobs(gated=True)
COMPS = ("idw", "rk_side", "krig_global", "kgnn", "gp_ml", "rk_onehot", "rk_spline", "rk_gbm", "rk_onehot_g", "rk_spline_g", "rk_gbm_g")
SRC_DENS = (SD["src_dens"][len(SD["src_dens"]) // 2],); PRE_KW = dict(dens=SRC_DENS, max_epochs=SD["max_epochs"])

covariates: 26 (simulator side information + 24 generated) | f = 32 interaction terms
f on an evaluation map: mean 0.07 sd 0.97 | linear R2 of f on the covariates 0.142 (must be small)


## 3. Stage 1 (development layouts), part A: D0 = the learnability gap

PASS needs (a) median corr(prior, f) >= 0.5 on held-out maps and (b) a gap of >= 0.25 to the *best* within-map comparator at `n_ref` labels. The same table at the large label budget shows whether within-map learning would catch up with more labels (a label-budget gap) or never does (a capacity gap). If D0 fails, do not run Stage 2: raise `D_EXTRA`, lower `RHO`, or raise `K`, and log it as v7.

In [4]:
PRIORS = f"{V6}/priors"                                  # priors only ever see SOURCE maps, so Stage 1 and Stage 2 share them
PRI = {}
for amp in SD["AMPS"]:
    PRI[amp] = x6.pretrain_prior_scale6(SRC, VAL, SD["Ks"], WALL_D_SRC, PRIORS, f"a{amp}", x6.with_amp(PLANT, amp), **PRE_KW)
K0 = SD["Ks"][-1]; ck = x5._load_prior(PRI[0.3][K0]); print(f"prior K={K0}, amp 0.3: val R2 on held-out SOURCE layouts = {ck['val_r2']:+.3f} ({ck['n_train_pts']} points)")
D0 = x6.d0_table(x6.with_amp(PLANT, 0.3), PRI[0.3][K0], dev_pairs, {**WALL_D, "plant6": PLANT}, n_labels=(SD["n_ref"], SD["n_big"]), n_units=min(8, len(dev_pairs)))
D0.to_csv(f"{V6}/D0.csv", index=False); display(D0.groupby("n_labels").median(numeric_only=True).round(3))
V = x6.d0_verdict(D0, n_ref=SD["n_ref"], min_prior=D0_MIN_PRIOR, min_gap=D0_MIN_GAP); print(json.dumps(V, indent=1))
print("D0:", "PASSED" if V["pass"] else "FAILED -> do NOT run Stage 2")
if not SMOKE: assert V["pass"], "D0 failed"

=== prior [a0.0] K=16: 64 source units, validation on 32 held-out source units
=== prior [a0.0] K=64: 256 source units, validation on 32 held-out source units
=== prior [a0.1] K=16: 64 source units, validation on 32 held-out source units
=== prior [a0.1] K=64: 256 source units, validation on 32 held-out source units
=== prior [a0.2] K=16: 64 source units, validation on 32 held-out source units
=== prior [a0.2] K=64: 256 source units, validation on 32 held-out source units
=== prior [a0.3] K=16: 64 source units, validation on 32 held-out source units
=== prior [a0.3] K=64: 256 source units, validation on 32 held-out source units
=== prior [a0.5] K=16: 64 source units, validation on 32 held-out source units
=== prior [a0.5] K=64: 256 source units, validation on 32 held-out source units
prior K=64, amp 0.3: val R2 on held-out SOURCE layouts = +0.477 (153600 points)


,corr_prior,linear_r2_f_on_X,corr_rk_side,corr_rk_onehot,corr_rk_spline,corr_rk_gbm
n_labels,,,,,,
100,0.548,0.168,0.065,0.060,0.080,0.065
1000,0.551,0.168,0.028,0.076,0.074,0.148


{
 "n_ref": 100,
 "median_corr_prior": 0.5482275796730872,
 "median_best_within": 0.09620950388977231,
 "gap": 0.4520180757833149,
 "median_linear_r2": 0.16780216183207997,
 "median_best_within_at_max_n": 0.14793462421294873,
 "n_max": 1000,
 "pass": true
}
D0: PASSED


## 4. Stage 1, part B: positive control on development layouts (all comparators incl. within-map nonlinear ones)

In [5]:
PL3 = x6.with_amp(PLANT, 0.3); WALL_DP = {**WALL_D, "plant6": PL3}
pj = x6.prior_jobs6(PRI[0.3], z=GATE_Z, rel=GATE_REL); learned_dev = [j["name"] for j in pj]
LEAK = x6.leak_check6(pj[:2] + [j for j in JOBS_WITHIN if j["name"].endswith("_g")], eval_names[0], WALL_DP, density=SD["pc_dens"][0]); print("leak check:", LEAK); assert all(LEAK.values()), "LEAK"
path_dev = f"{V6}/dev_plant.jsonl"
x6.run_tasks6(x5.make_tasks5(eval_names, SD["pc_dens"], SD["reps"][:1], JOBS_CLASSICAL + JOBS_WITHIN + pj, WALL_DP, "V6dev"), path_dev, N_JOBS)
x5.check_run(path_dev, strict=not SMOKE); dDev = rc.read_results(path_dev)
comps = [c for c in COMPS if c in set(dDev.name)]; cDev, wDev = x5.claim_cells(dDev, learned_dev, comps, "wall6_dev", MARGIN_PCT)
wDev["gated"] = ~wDev.A.str.endswith("_ng"); display(wDev.sort_values("worst_ci_lo", ascending=False).round(3))
print("accept rate of the gate:"); display(x5.accept_rate(dDev, [n for n in learned_dev if not n.endswith("_ng")]).round(2))
DEV_PASS = bool(wDev[wDev.gated & wDev.clears_margin_vs_all].shape[0] > 0)
print("STAGE-1 POSITIVE CONTROL:", "PASSED" if DEV_PASS else "FAILED -> do NOT run Stage 2")
if not SMOKE: assert DEV_PASS, "Stage-1 control failed"

leak check: {'prior6_K16': True, 'prior6_K16_ng': True, 'rk_onehot_g': True, 'rk_spline_g': True, 'rk_gbm_g': True}
[run_tasks6] 192 tasks, 122 done, 70 to run (n_jobs=15) -> results_ext5/v6/dev_plant.jsonl
  25/70 708s failed jobs: 0
  50/70 1236s failed jobs: 0
  70/70 1521s failed jobs: 0
[check_run] results_ext5/v6/dev_plant.jsonl: 3072 rows, 0 failed (0.00%)


,A,density,worst_ci_lo,worst_gain_pct,best_gain_pct,min_ci_hi,n_comp,clears_margin_vs_all,excluded,gated
9,prior6_K64_ng,100,19.109,21.180,25.809,22.761,11,True,False,False
6,prior6_K64,100,17.115,19.163,23.799,21.059,11,True,False,True
10,prior6_K64_ng,300,16.946,18.338,25.107,19.718,11,True,False,False
7,prior6_K64,300,16.085,17.578,24.417,19.076,11,True,False,True
3,prior6_K16_ng,100,14.942,17.479,22.318,19.468,11,True,False,False
8,prior6_K64,1000,14.213,15.816,25.604,17.478,11,True,False,True
11,prior6_K64_ng,1000,14.213,15.816,25.604,17.478,11,True,False,False
4,prior6_K16_ng,300,12.465,13.786,20.941,15.000,11,True,False,False
1,prior6_K16,300,11.396,12.860,20.124,14.273,11,True,False,True
0,prior6_K16,100,10.186,13.259,18.421,15.909,11,True,False,True


accept rate of the gate:


density,100,300,1000
name,,,
prior6_K16,0.70,0.86,0.97
prior6_K64,0.88,0.92,1.00


STAGE-1 POSITIVE CONTROL: PASSED


## 5. Freeze the plan (write-once, hashed) BEFORE the fresh layouts are built

Claim rules: **positive (semi-synthetic)**: at some `amp > 0` and K, the *gated* prior beats every comparator in `COMPS` (including the three nonlinear within-map ones) with layout-cluster CI lower bound > 2% on fresh layouts. **False-positive control**: at `amp = 0` the gated prior is equivalent to `rk_side` (CI within +/-2%). **Crossover**: the smallest `amp` at which the claim holds, per K and density. Commit `prereg_ext6.json` to git before running Stage 2.

In [6]:
PLAN6 = dict(title="round 6: learnability gap, nonlinear within-map comparators, global raw plant, fresh layouts", parent="prereg_ext5_amend_v4.json",
             changes=["within-map comparators rk_onehot/rk_spline/rk_gbm (+ CV-gated _g)", "planted f on globally scaled raw covariates", "d_extra=%d generated covariates, rho=%.2f" % (D_EXTRA, RHO),
                      "D0 = learnability gap (min prior corr %.2f, min gap %.2f)" % (D0_MIN_PRIOR, D0_MIN_GAP), "fresh layouts, seed ledger"],
             plant={k: v for k, v in PLANT.items() if k not in ("mu", "sd")}, plant_constants_sha=hashlib.sha256(json.dumps([PLANT["mu"], PLANT["sd"], PLANT["f_mu"], PLANT["f_sd"]]).encode()).hexdigest(),
             fresh=dict(pool_seed=FRESH_POOL_SEED, sim_seed=FRESH_SIM_SEED, n_eval=SD["n_eval"], n_tx=SD["n_tx"], n_pts=SD["n_pts"]), amps=list(SD["AMPS"]), Ks=list(SD["Ks"]), densities=list(SD["pc_dens"]), reps=SD["reps"][:1],
             comparators=list(COMPS), margin_pct=MARGIN_PCT, gate=dict(z=GATE_Z, rel=GATE_REL, folds=5, min_labels=30), d0=V,
             prior=dict(hidden=[128, 128, 128], alpha=1e-4, lr=2e-3, batch=2048, max_epochs=SD["max_epochs"], patience=8, source_density=list(SRC_DENS), max_pts_per_map=600))
print("plan sha256:", x6.freeze_plan(PLAN6, f"{ROOT}/prereg_ext6.json")[:16])

plan sha256: 0141caf74b0bf8a4


## 6. Stage 2 (FRESH layouts, run once): dose-response over planted amplitude, amp = 0 is the false-positive control

In [7]:
used = dev_pairs + SRC_PAIRS + VAL_PAIRS
old_fresh = f"{ROOT}/cache_scale_fresh"
if os.path.isdir(old_fresh):
    WALL_F_OLD = {**WALL_D, "cache_dir": old_fresh, "test_salt": "scale5fresh", "seed": 7}; nm, _ = x5.build_scale_pool(WALL_F_OLD, SD["n_eval"], 0, SD["n_tx"], SD["n_pts"], seed=21); used += [(n, old_fresh) for n in nm]
WALL_F, fresh_names, fresh_pairs = x6.fresh_pool(WALL_D, ROOT, "scale6fresh", FRESH_POOL_SEED, FRESH_SIM_SEED, SD["n_eval"], SD["n_tx"], SD["n_pts"], used, LEDGER)
print(len(fresh_names), "FRESH evaluation units (disjoint by fingerprint from", len(used), "layouts used so far)")
CONF = {}
for amp in SD["AMPS"]:
    pl = x6.with_amp(PLANT, amp); pjc = x6.prior_jobs6(PRI[amp], z=GATE_Z, rel=GATE_REL); path = f"{V6}/confirm_a{amp}.jsonl"
    x6.run_tasks6(x5.make_tasks5(fresh_names, SD["pc_dens"], SD["reps"][:1], JOBS_CLASSICAL + JOBS_WITHIN + pjc, {**WALL_F, "plant6": pl}, f"V6c{amp}"), path, N_JOBS)
    x5.check_run(path, strict=not SMOKE); CONF[amp] = (rc.read_results(path), [j["name"] for j in pjc])
x6.spend_seeds(LEDGER, FRESH_POOL_SEED, FRESH_SIM_SEED); print("fresh seeds are now burned:", open(LEDGER).read())

64 FRESH evaluation units (disjoint by fingerprint from 1184 layouts used so far)
[run_tasks6] 192 tasks, 0 done, 192 to run (n_jobs=15) -> results_ext5/v6/confirm_a0.0.jsonl
  25/192 576s failed jobs: 0
  50/192 1101s failed jobs: 0
  75/192 1631s failed jobs: 0
  100/192 2132s failed jobs: 0
  125/192 2633s failed jobs: 0
  150/192 3133s failed jobs: 0
  175/192 3639s failed jobs: 0
  192/192 3834s failed jobs: 0
[check_run] results_ext5/v6/confirm_a0.0.jsonl: 3072 rows, 0 failed (0.00%)
[run_tasks6] 192 tasks, 0 done, 192 to run (n_jobs=15) -> results_ext5/v6/confirm_a0.1.jsonl
  25/192 576s failed jobs: 0
  50/192 1096s failed jobs: 0
  75/192 1622s failed jobs: 0
  100/192 2132s failed jobs: 0
  125/192 2643s failed jobs: 0
  150/192 3135s failed jobs: 0
  175/192 3646s failed jobs: 0
  192/192 3846s failed jobs: 0
[check_run] results_ext5/v6/confirm_a0.1.jsonl: 3072 rows, 0 failed (0.00%)
[run_tasks6] 192 tasks, 0 done, 192 to run (n_jobs=15) -> results_ext5/v6/confirm_a0.2.jsonl

## 7. Dose-response, false-positive control, claim

In [8]:
rows, ctabs = [], []
for amp, (d, learned) in CONF.items():
    comps = [c for c in COMPS if c in set(d.name)]; c, w = x5.claim_cells(d, learned, comps, f"wall6_a{amp}", MARGIN_PCT); c["amp"] = amp; w["amp"] = amp
    ar = x5.accept_rate(d, [n for n in learned if not n.endswith("_ng")]); w["accept_rate"] = [ar.loc[a, dd] if (a in ar.index and dd in ar.columns) else np.nan for a, dd in zip(w.A, w.density)]
    ctabs.append(c); rows.append(w)
W = pd.concat(rows, ignore_index=True); W["K"] = W.A.str.extract(r"_K(\d+)")[0].astype(int); W["gated"] = ~W.A.str.endswith("_ng"); C = pd.concat(ctabs, ignore_index=True)
W.to_csv(f"{V6}/confirm_claim_cells.csv", index=False); C.to_csv(f"{V6}/confirm_contrasts.csv", index=False)
print("worst_gain_pct = mean gain vs the STRONGEST comparator (gated arm); clears = CI lower bound > margin vs ALL comparators")
display(W[W.gated].pivot_table(index=["K", "density"], columns="amp", values="worst_gain_pct").round(2)); display(W[W.gated].pivot_table(index=["K", "density"], columns="amp", values="clears_margin_vs_all", aggfunc="first"))
print("accept rate of the gate:"); display(W[W.gated].pivot_table(index=["K", "density"], columns="amp", values="accept_rate").round(2))
print("\nWHICH COMPARATOR IS THE STRONGEST (amp 0.3, gated arm, largest K):")
Cb = C[(C.amp == 0.3) & (~C.A.str.endswith("_ng")) & (C.A.str.endswith(f"_K{max(SD['Ks'])}"))]; display(Cb.pivot_table(index="B", columns="density", values="mean_gain_pct").round(2))
if 0.0 in CONF:
    fp = C[(C.amp == 0.0) & (C.B == "rk_side")][["A", "density", "mean_gain_pct", "ci_lo", "ci_hi", "verdict"]]; print("\nFALSE-POSITIVE CONTROL (amp = 0, vs rk_side):"); display(fp.round(3))
    print("gate does no harm at amp = 0:", bool(fp[~fp.A.str.endswith("_ng")].verdict.str.startswith("equivalent").all()))
pos = W[W.gated & (W.amp > 0) & W.clears_margin_vs_all]; print("\nPOSITIVE CELLS on fresh layouts (gated prior beats EVERY comparator by > margin):", len(pos))
if len(pos): display(pos.groupby(["K", "density"]).amp.min().rename("smallest_amp_that_wins").reset_index())

worst_gain_pct = mean gain vs the STRONGEST comparator (gated arm); clears = CI lower bound > margin vs ALL comparators


amp          0.0   0.1   0.2    0.3    0.5
K  density                                
16 100      2.06  1.89  6.56  14.07  23.94
   300      0.00 -0.08  2.78  11.12  24.83
   1000    -0.08  0.00  3.95  11.30  21.41
64 100      2.56  4.06  8.42  17.67  32.26
   300      0.20  0.33  5.85  18.27  33.64
   1000    -0.08  0.09  7.92  17.22  31.86

amp           0.0    0.1    0.2   0.3   0.5
K  density                                 
16 100      False  False   True  True  True
   300      False  False  False  True  True
   1000     False  False   True  True  True
64 100      False  False   True  True  True
   300      False  False   True  True  True
   1000     False  False   True  True  True

accept rate of the gate:


amp          0.0   0.1   0.2   0.3   0.5
K  density                              
16 100      0.30  0.23  0.45  0.72  0.84
   300      0.00  0.02  0.34  0.75  0.98
   1000     0.00  0.00  0.66  0.94  1.00
64 100      0.28  0.39  0.53  0.75  0.94
   300      0.03  0.11  0.55  0.95  0.98
   1000     0.00  0.09  0.77  0.98  1.00

...

density,100,300,1000
B,,,
gp_ml,19.72,23.27,26.35
idw,19.42,23.87,26.38
kgnn,18.06,22.99,25.44
krig_global,18.70,23.36,26.42
rk_gbm,18.07,19.19,17.22
rk_gbm_g,17.79,18.44,19.12
rk_onehot,23.96,26.19,22.32
rk_onehot_g,17.68,18.33,20.00
rk_side,17.68,18.33,20.00



FALSE-POSITIVE CONTROL (amp = 0, vs rk_side):


,A,density,mean_gain_pct,ci_lo,ci_hi,verdict
3,prior6_K16,100,2.057,0.982,3.234,A better
4,prior6_K16,300,0.000,0.000,0.000,equivalent (CI within +/-2%)
5,prior6_K16,1000,0.000,0.000,0.000,equivalent (CI within +/-2%)
36,prior6_K16_ng,100,1.485,-0.623,3.444,inconclusive
37,prior6_K16_ng,300,-6.909,-8.208,-5.621,A worse
38,prior6_K16_ng,1000,-9.770,-11.082,-8.454,A worse
69,prior6_K64,100,2.561,1.397,3.778,A better
70,prior6_K64,300,0.195,0.000,0.536,equivalent (CI within +/-2%)
71,prior6_K64,1000,0.000,0.000,0.000,equivalent (CI within +/-2%)
102,prior6_K64_ng,100,3.278,1.094,5.308,A better


gate does no harm at amp = 0: False

POSITIVE CELLS on fresh layouts (gated prior beats EVERY comparator by > margin): 17


,K,density,smallest_amp_that_wins
0,16,100,0.2
1,16,300,0.3
2,16,1000,0.2
3,64,100,0.2
4,64,300,0.2
5,64,1000,0.2


## 8. What you may write

| Outcome | Wording the data allow |
|---|---|
| D0 passed, Stage 1 passed, Stage 2 has positive cells, amp = 0 control equivalent | *"On a semi-synthetic benchmark with shared high-dimensional side-information structure that a single map cannot learn from ~100 labels (D0), a pre-trained prior combined with kriging beats classical and within-map nonlinear regression-kriging once amplitude exceeds a* and K >= K*; without shared structure the gate returns rk_side."* Report the dose-response table and name the generated covariates as generated. |
| D0 failed | No Stage 2. The setting does not separate cross-layout from within-map learning; change `D_EXTRA`/`RHO`/K and log as v7. |
| Stage 2 positive cells exist but the strongest comparator is `rk_gbm_g` or `rk_onehot_g` in most cells | Say the gain over classical arms is mostly available within a map; report only the margin over those. |
| Stage 2 has no positive cell | Report it. Do not re-seed the fresh layouts; the ledger now blocks that. |

**Still limiting.** This says *when* transfer can help in a simulator with planted structure. It is not evidence that GNNs (or priors) help on real radio maps; that needs a real dataset with shared side-information structure evaluated leave-layout-out with the same gate. Do not select the best amp/K after the fact.